In [1]:
import warnings
warnings.filterwarnings("ignore")
import good
import json
import os
import pandas as pd
import importlib
from good import s_runner
from good import helper
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from pathlib import Path

In [2]:
# =============================================================================
# Input settings
# =============================================================================
N_SCENARIO_WORKERS = 5
CPLEX_THREADS_PER_SCENARIO = 50

IPM_REGION = "FRCC"
GOOD_ROOT = Path.home() / "GOOD_MODEL"
EV_DATA_PATH = GOOD_ROOT / "Examples" / "EVDATA" / f"{IPM_REGION}_EVDATA.json"

with open(EV_DATA_PATH, "r") as f:
    ev_data = json.load(f)


YEAR = 2030

SCENARIOS = helper.load_scenarios(IPM_REGION, YEAR)

YEAR_INPUT = 2030
MONTH_INPUT = 0
DAY_INPUT = 360
SEASON = "Annual"
STATE = "FL"
BASE_GRAPH = good.graph.graph_from_json("Examples/Nodes/Florida_IPM.json")
BASE_POLICIES = good.utilities.read_json("Examples/policies.json")
TARGET_PEAK_GW = None  # set this to your Florida/FRCC target peak if you use peak scaling
Discount_rate = 0.07
Lifetime = 25

ADOPTION_SCENARIOS = [
                      "slow",
                      "mid",
                      "fast"
                      ]
# =============================================================================
# Configuration
# =============================================================================
# Define all charging scenarios
CHARGING_SCENARIOS = {
    "midnight": {
        "profile": "timed_charging",
        "description": "Midnight timed charging",
    },
    "delay": {
        "profile": "max_delay",
        "description": "Maximum delay charging",
    },
    "arrive": {
        "profile": "min_delay",
        "description": "Immediate arrival charging",
    },
    "flex": {
        "profile": "load_leveling",
        "description": "Flexible load leveling",
    },
}

# Battery storage distribution weights
BATTERY_WEIGHTS = {
    "FRCC": 1.0,
}

assert abs(sum(BATTERY_WEIGHTS.values()) - 1.0) < 1e-6

FRCC_REGIONS = [
    "FRCC",
]

STATE_TO_REGIONS = {
    "FL": FRCC_REGIONS,
    "FRCC": FRCC_REGIONS,
}

RETIREMENT_POLICIES = {
    "TX": {
        "coal": 1 / 3,
    },

    "NY": {
        "oil": 0.10,
        "natural gas turbine": 0.15,
    },

    "CA": {
        "oil": 0.70,
        "nuclear": 1.00,
        "natural gas turbine": 0.20,
        "natural gas combined cycle": 0.04,
        "coal": 1.00,
    },

    "FL": {
        # Start conservative.
        # You can update later based on Florida retirement assumptions.
        "coal": 0.20,
        "oil": 0.20,
    },

}
ASSET_CONSTRAINT_POLICIES = {
    "TX": {
        # ERCOT 2030 central logic:
        # Nuclear is small in capacity but high in energy output, so keep it highly must-run.
        # Coal is still meaningful, but do NOT force it at 45% in a 2030 renewable-heavy case.
        # Combined-cycle gas should provide reliability, but not behave as baseload in every hour.

        "nuclear": {
            "must_run_fraction": 0.90,
            "ramp_rate": 0.03,
        },

        "coal": {
            "must_run_fraction": 0.25,
            "ramp_rate": 0.05,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.12,
            "ramp_rate": 0.25,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.80,
        },

        "oil": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.30,
        },
    },

    "NY": {
        # NYISO 2030 central logic:
        # Existing nuclear should be almost always online.
        # Coal should not have a must-run floor. NY has effectively moved away from coal.
        # Gas/oil and dual-fuel units are important for reliability, especially downstate and winter,
        # but they should not be forced as baseload in a clean-transition scenario.

        "nuclear": {
            "must_run_fraction": 0.95,
            "ramp_rate": 0.03,
        },

        "coal": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.05,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.08,
            "ramp_rate": 0.20,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.80,
        },

        "oil": {
            "must_run_fraction": 0.05,
            "ramp_rate": 0.20,
        },
    },

    "CA": {
        # CAISO / California 2030 central logic:
        # Do not add a nuclear constraint unless your GOOD input data explicitly includes CA nuclear.
        # California policy still treats Diablo Canyon as a transition/reliability resource, not a long-run
        # 2030+ baseload expansion resource.
        # Gas must stay flexible because CA has solar-heavy duck-curve conditions.

        # If your GOOD data has a CA "nuclear" category and you are modeling a 2030 week
        # before Diablo Canyon Unit 2 retirement, you may add this sensitivity:
        #
        # "nuclear": {
        #     "must_run_fraction": 0.60,
        #     "ramp_rate": 0.03,
        # },

        "coal": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.08,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.05,
            "ramp_rate": 0.25,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.80,
        },

        "oil": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.30,
        },
    },

    "FL": {
        # Florida / FRCC 2030 central logic:
        # Nuclear is baseload.
        # Florida relies heavily on natural gas, especially combined-cycle gas.
        # Coal still exists but should have a low floor because coal retirements and solar growth continue.
        # Oil should not be must-run.

        "nuclear": {
            "must_run_fraction": 0.95,
            "ramp_rate": 0.03,
        },

        "coal": {
            "must_run_fraction": 0.08,
            "ramp_rate": 0.04,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.18,
            "ramp_rate": 0.18,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.60,
        },

        "oil": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.30,
        },
    },

    "PJM": {
        # PJM 2030 central logic:
        # Nuclear is large and should remain highly must-run.
        # Coal is still important, but a 45% hourly floor is too high for a 2030 policy/RPS case.
        # Combined-cycle gas is important for reliability and load growth, but it should not be forced
        # as baseload in every hour.

        "nuclear": {
            "must_run_fraction": 0.95,
            "ramp_rate": 0.03,
        },

        "coal": {
            "must_run_fraction": 0.25,
            "ramp_rate": 0.05,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.15,
            "ramp_rate": 0.20,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.80,
        },

        "oil": {
            "must_run_fraction": 0.05,
            "ramp_rate": 0.25,
        },
    },
}

FLEXIBILITY_POLICIES = {
    "default": {
        # -----------------------------
        # V1G settings
        # -----------------------------
        "v1g": {
            "base_shift_cost": 0.0,
            "fixed_om_per_kw_year": 0.0,
            "shift_window_hours": 24,
        },

        # -----------------------------
        # V2G settings
        # -----------------------------
        "v2g": {
            "window_hours": 24,
            "energy_duration_hours": 1,
            "roundtrip_efficiency": 0.985,
            "base_shift_cost": 1.3784e-9,
            "fixed_om_per_kw_year": 0.0,
        },

        # -----------------------------
        # Stationary battery settings
        # -----------------------------
        "battery": {
            "duration_hours": 4,
            "charge_efficiency": 0.93,
            "discharge_efficiency": 0.92,
            "fixed_om_per_kw_year": 3.75,
            "cycling_cost_per_mwh": 0.01,
            "initial_soc_fraction": 0.50,

            # Default total stationary battery power
            # You can override by state below.
            "total_power_mw": 600,
        },
    },

    "CA": {
        "battery": {
            "total_power_mw": 16900,
        },
    },

    "NY": {
        "battery": {
            "total_power_mw": 690,
        },
    },

    "TX": {
        "battery": {
            "total_power_mw": 17500,
        },

    "FL": {
        "battery": {
            "total_power_mw": 600,
        },
        },
    },
}

ECONOMIC_POLICIES = {
    "default": {
        "discount_rate": 0.07,
        "lifetime_years": 25,
        "import_operating_cost": 1.75e-8,
        "apply_crf_to_renewables": True,
        "apply_crf_to_storage": True,
        "renewable_fuels_for_crf": {"solar", "wind"},
    },
    "NY": {},
    "CA": {},
    "TX": {},
    "FL": {},
}

TRANSMISSION_POLICIES = {
    "default": {
        "apply_distance_enhancement": True,
        "default_operating_cost": 2.222222222222222e-09,
        "default_efficiency": 0.90,
        # If False, only fill missing or zero line operating costs.
        # If True, overwrite all line operating costs.
        "overwrite_existing_operating_cost": False,
        # If False, only fill missing efficiencies.
        # If True, overwrite all line efficiencies.
        "overwrite_existing_efficiency": False,
    },
    "NY": {},
    "CA": {},
    "TX": {},
    "FL": {},
}

In [ ]:
"""
Professional Scenario Runner - Loops Through All Adoption × Charging Scenarios
Runs all combinations of adoption levels and charging patterns separately.
Each combination gets its own folder.
"""
# =============================================================================
# Main Loop - Process Each Adoption × Charging Scenario Combination
# =============================================================================

importlib.reload(s_runner)


total_combinations = len(ADOPTION_SCENARIOS) * len(CHARGING_SCENARIOS)
total_runs = len(SCENARIOS) * total_combinations
print(f"\n{'#'*80}")
print(f"# RUNNING ALL ADOPTION × CHARGING SCENARIOS")
print(f"# Adoption scenarios: {len(ADOPTION_SCENARIOS)} ({', '.join(ADOPTION_SCENARIOS)})")
print(f"# Charging scenarios: {len(CHARGING_SCENARIOS)} ({', '.join(CHARGING_SCENARIOS.keys())})")
print(f"# Total combinations: {total_combinations}")
print(f"# Total runs: {total_runs} (= {len(SCENARIOS)} scenarios × {total_combinations} combinations)")
print(f"{'#'*80}\n")
# Track overall progress
all_results_tracker = {}

for adoption_level in ADOPTION_SCENARIOS:

    print(f"\n{'█'*80}")
    print(f"█ ADOPTION LEVEL: {adoption_level.upper()}")
    print(f"{'█'*80}\n")

    for charging_name, charging_config in CHARGING_SCENARIOS.items():
        PROJECT_ROOT = "/ocean/projects/ele250005p/htayarani/GOOD_MODEL"

        RESULTS_DIR = (
            f"{PROJECT_ROOT}/Output/{IPM_REGION}/"f"scenario_results_{YEAR_INPUT}_{adoption_level}_{IPM_REGION}_{charging_name}")
        # Set up this combination
        # RESULTS_DIR = f"Output/{IPM_REGION}/scenario_results_{YEAR_INPUT}_{adoption_level}_{IPM_REGION}_{charging_name}"
        CHARGING_PROFILE = charging_config["profile"]


        ctx = s_runner.ScenarioRunContext(
            scenarios=SCENARIOS,
            base_graph=BASE_GRAPH,
            base_policies=BASE_POLICIES,
            state_to_regions=STATE_TO_REGIONS,
            ev_data=ev_data,
            battery_weights=BATTERY_WEIGHTS,
            results_dir=RESULTS_DIR,
            discount_rate=Discount_rate,
            lifetime=Lifetime,
            retirement_policies=RETIREMENT_POLICIES,
            asset_constraint_policies=ASSET_CONSTRAINT_POLICIES,
            solver_threads=CPLEX_THREADS_PER_SCENARIO,
        )
        # Create unique key for tracking
        combo_key = f"{adoption_level}_{charging_name}"

        print(f"\n{'='*80}")
        print(f"COMBINATION: {adoption_level.upper()} adoption + {charging_config['description']}")
        print(f"Results Directory: {RESULTS_DIR}")
        print(f"{'='*80}\n")

        # Create results directory
        os.makedirs(RESULTS_DIR, exist_ok=True)

        # Run all scenarios for this combination
        all_results = []
        failed_scenarios = []

        tasks = []

        for scenario_id in sorted(SCENARIOS.keys()):
            tasks.append({
                "scenario_id": scenario_id,
                "ctx": ctx,
                "year": YEAR_INPUT,
                "adoption": adoption_level,
                "charging": CHARGING_PROFILE,
                "charging_name": charging_name,
                "charging_description": charging_config["description"],
                "month": MONTH_INPUT,
                "day_duration": DAY_INPUT,
                "model_region": IPM_REGION,
                "discount_rate": Discount_rate,
                "lifetime": Lifetime,
                "fix_peak": False,
                "target_peak_gw": TARGET_PEAK_GW,
                "peak_region_mode": "all",
                "peak_regions": None,
                "retirement_policy": None,
                "asset_constraint_policy": None,
                "use_state_default_retirement": True,
                "use_state_default_asset_constraints": True,
                "flexibility_policy": FLEXIBILITY_POLICIES,
                "use_state_default_flexibility": True,
                "economic_policy": ECONOMIC_POLICIES,
                "use_state_default_economic_policy": True,
                "transmission_policy": TRANSMISSION_POLICIES,
                "use_state_default_transmission_policy": True,
            })

        mp_context = mp.get_context("spawn")

        with ProcessPoolExecutor(
            max_workers=N_SCENARIO_WORKERS,
            mp_context=mp_context,
        ) as executor:

            futures = {
                executor.submit(s_runner.run_one_scenario_parallel_task, task): task["scenario_id"]
                for task in tasks
            }

            for future in as_completed(futures):
                scenario_id = futures[future]
                result = future.result()

                if result["ok"]:
                    all_results.append(result["row"])
                    print(f"  Scenario {scenario_id:>3} ✓")
                else:
                    print(f"  Scenario {scenario_id:>3} ✗ Error: {result['error']}")

                    failed_scenarios.append({
                        "scenario_id": scenario_id,
                        "error": result["error"],
                        "traceback": result["traceback"],
                    })

        # Save results for this combination
        all_results_df = pd.DataFrame(all_results)
        summary_path = os.path.join(RESULTS_DIR, "all_scenarios_summary.csv")
        all_results_df.to_csv(summary_path, index=False)

        # Save failed scenarios if any
        if failed_scenarios:
            failed_df = pd.DataFrame(failed_scenarios)
            failed_path = os.path.join(RESULTS_DIR, "failed_scenarios.csv")
            failed_df.to_csv(failed_path, index=False)

        # Store for later comparison
        all_results_tracker[combo_key] = all_results_df

        # Print summary for this combination
        print(f"\n  Summary:")
        print(f"    Successful: {len(all_results)}/{len(SCENARIOS)}")
        print(f"    Failed: {len(failed_scenarios)}/{len(SCENARIOS)}")
        print(f"    Saved to: {summary_path}")

        if failed_scenarios:
            print(f"    ⚠ Failed scenarios logged to: {failed_path}")


################################################################################
# RUNNING ALL ADOPTION × CHARGING SCENARIOS
# Adoption scenarios: 3 (slow, mid, fast)
# Charging scenarios: 4 (midnight, delay, arrive, flex)
# Total combinations: 12
# Total runs: 180 (= 15 scenarios × 12 combinations)
################################################################################


████████████████████████████████████████████████████████████████████████████████
█ ADOPTION LEVEL: SLOW
████████████████████████████████████████████████████████████████████████████████


COMBINATION: SLOW adoption + Midnight timed charging
Results Directory: /ocean/projects/ele250005p/htayarani/GOOD_MODEL/Output/FRCC/scenario_results_2030_slow_FRCC_midnight

EV state: FL
Model region: FRCC
STATE_TO_REGIONS key used: FRCC
GOOD regions used: ['FRCC']
Base load peak scaling is OFF. Using original graph load.
Created RPS policies:
  rps_FRCC: ratio=0.5, regions=['FRCC']
Total battery assets removed: 1

EV profil